# Apêndice · Devem os gráficos de longo prazo ser ajustados para a inflação? O caso brasileiro
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**: Ibovespa e dólar do Yahoo Finance (^BVSP e BRL=X) e IPCA mensal do Banco Central (SGS 433). Na primeira execução, o notebook baixa esses dados na sua máquina e guarda uma cópia local (`dados_apendice.pkl`), que as execuções seguintes reaproveitam. O Yahoo corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. Sem custos nem impostos; não é recomendação.

In [ ]:
!pip install -q yfinance

In [ ]:
import os
import json
import urllib.request
import numpy as np
import pandas as pd
import yfinance as yf

FIM = "2026-10-01"
CACHE = "dados_apendice.pkl"


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


if os.path.exists(CACHE):
    ibov, usd, ipca = pd.read_pickle(CACHE)
else:
    ibov = yf.download("^BVSP", start="1995-01-01", end=FIM, auto_adjust=False, progress=False, multi_level_index=False)["Close"]
    usd = yf.download("BRL=X", start="2003-01-01", end=FIM, auto_adjust=False, progress=False, multi_level_index=False)["Close"]
    confere({"^BVSP": ibov, "BRL=X": usd})
    url = ("https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json"
           "&dataInicial=01/01/1995&dataFinal=30/09/2026")
    d = json.load(urllib.request.urlopen(url, timeout=30))
    ipca = pd.Series([float(x["valor"]) for x in d], index=pd.to_datetime([x["data"] for x in d], dayfirst=True))
    pd.to_pickle((ibov, usd, ipca), CACHE)
pc = lambda x, k=0: f"{x:.{k}%}".replace(".", ",")
num = lambda x: f"{x:,.0f}".replace(",", ".")

# índice de preços acumulado, com o IPCA do mês aplicado ao fim do mês
indice = (1 + ipca / 100).cumprod()
indice.index = indice.index + pd.offsets.MonthEnd(0)
fator = indice.reindex(ibov.index, method="ffill").bfill()
fator = fator.iloc[-1] / fator                                 # reais de setembro/2026
real = ibov * fator
dol = (ibov / usd.reindex(ibov.index, method="ffill")).dropna()

## Parte 1: Ibovespa nominal, descontado pelo IPCA e em dólares

In [ ]:
print("Parte 1. Ibovespa de 1995 a set/2026")
ini, fim = ibov.index[0], ibov.index[-1]
vz = lambda x: f"{x:.1f}".replace(".", ",")
print(f"   nominal: {num(ibov.iloc[0])} -> {num(ibov.iloc[-1])} ({vz(ibov.iloc[-1] / ibov.iloc[0])} vezes)")
print(f"   IPCA acumulado no período: {pc(indice.iloc[-1] / indice.iloc[0] - 1)}")
print(f"   real (em reais de set/2026): {num(real.iloc[0])} -> {num(real.iloc[-1])} ({vz(real.iloc[-1] / real.iloc[0])} vezes)")

## Parte 2: o topo de maio de 2008 em cada série

In [ ]:
print("\nParte 2. O topo de maio de 2008 e quando cada série passou dele")
for rot, s in (("nominal", ibov), ("real (IPCA)", real), ("em dólares", dol)):
    s08 = s.loc["2008-01-01":"2008-12-31"]
    topo, dt = s08.max(), s08.idxmax()
    depois = s.loc[dt:]
    acima = depois[depois > topo]
    quando = (f"passou em {acima.index[0]:%m/%Y} ({vz((acima.index[0] - dt).days / 365.25)} anos depois)"
              if len(acima) else "ainda não passou")
    print(f"   {rot:12s}: topo de 2008 = {num(topo)} em {dt:%d/%m/%Y}; {quando};"
          f" set/2026 = {num(s.iloc[-1])} ({pc(s.iloc[-1] / topo - 1)} do topo)")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4))
for s, rot in ((ibov, "nominal"), (real, "descontado pelo IPCA")):
    ax.plot(s.index, s, label=rot)
    ax.axhline(s.loc["2008"].max(), ls="--", lw=0.8)
ax.set_yscale("log")
ax.set_title("Ibovespa nominal e real, com o topo de maio de 2008")
ax.legend()
plt.show()

## Parte 3: o topo nominal de 2008 virou referência?

In [ ]:
topo_nom = ibov.loc["2008"].max()
faixa = ibov.loc["2009":][(ibov.loc["2009":] > topo_nom * 0.97) & (ibov.loc["2009":] <= topo_nom * 1.03)]
meses = faixa.index.to_period("M").unique()
print(f"\nParte 3. Fechamentos a até 3% do topo nominal de 2008 ({num(topo_nom)}) depois de 2009: {len(faixa)} pregões em {len(meses)} meses")
print(f"   primeiro: {faixa.index[0]:%m/%Y}; último: {faixa.index[-1]:%m/%Y}")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a célula de abertura. Mostre onde o IPCA do mês é aplicado no fim do mês (`pd.offsets.MonthEnd(0)`) e por que `fator = fator.iloc[-1] / fator` deixa a série `real` em reais de setembro de 2026." Confira com a Parte 1, que imprime o IPCA acumulado de 644% e a alta real de 5,8 vezes, e com a Parte 2, que imprime o topo de 2008 descontado: 202.909 pontos.
2. **Adaptar:** "Troque o ticker `^BVSP` da célula de abertura por [ATIVO], cotado em reais e com cotação desde antes de 2008, com o ticker do Yahoo Finance, e dê outro nome ao arquivo `CACHE`, para não ler a cópia do Ibovespa. Não mude o desconto pelo IPCA nem a faixa de 3% da Parte 3." Antes de trocar, rode o notebook como está e confira se a Parte 3 continua dando 106 pregões em 16 meses, como no livro.

Mais pedidos deste apêndice (entender e aplicar ao seu gráfico) estão na página do apêndice, no hub do livro.

## Para ir além
1. Troque o IPCA pelo IGP-M (SGS 189). O topo de 2008 já foi superado em termos reais?
2. Faça a conta com o S&P 500 em dólares e o CPI americano. Murphy tinha razão para o mercado dele?
3. Na Parte 3, troque a faixa de 3% por 5%. Quantos pregões o topo de 2008 segurou?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.